# EXPLORATORIO

# U-Net 3D Denoising for Mock Cubes

# 0. Importaciones y Configuración

In [1]:
import sys
sys.path.append('../../src')

from astropy.io import fits
import torch
import torch.nn.functional as F
from u_net_model import UNet3D, UNet3DWithPadCrop
import numpy as np
import os
import matplotlib.pyplot as plt
import astropy.units as u
from radio_beam import Beams, Beam
from astropy.convolution import convolve, convolve_fft
from spectral_cube import SpectralCube

mkdir -p failed for path /.matplotlib: [Errno 30] Read-only file system: '/.matplotlib'
Matplotlib created a temporary cache directory at /var/folders/2b/5tqr_dzn551f21glw27fr5t40000gn/T/matplotlib-uqsn025f because there was an issue with the default path (/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


In [2]:
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Dispositivo: {device}')

WEIGHTS_PATH = '../../weights/weights_best.pt'
DATA_DIR = '/Users/kuky/Documents/practica/Denoiser3D-IFU/data/mock_cubes'

Dispositivo: mps


# 1. Definición de funciones

In [10]:
TRAINING_TARGET_SHAPE = (48, 96, 96)

def load_unet(weights_path, device, target_shape=TRAINING_TARGET_SHAPE):
    """Load U-Net with the same target_shape used during training.
    
    Parameters
    ----------
    weights_path : str
        Path to the saved model weights (.pt file).
    device : torch.device
        Device to load the model onto.
    target_shape : tuple of int, optional
        Fixed (D, H, W) used during training. Must match the shape the model
        was trained with so that internal feature scales are consistent.
        
    Returns
    -------
    model : UNet3DWithPadCrop
        Loaded model in eval mode. The wrapper handles padding/cropping
        automatically, so any input shape smaller than target_shape is valid.
    """
    base_model = UNet3D(n_channels=1, filters=16) # Se llama a la función UNet3D con los parámetros por default.
    model = UNet3DWithPadCrop(base_model, target_shape=target_shape)
    state_dict = torch.load(weights_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


def denoise_simple(cube, model, device):
    """Denoise by passing the cube directly through the bare UNet3D.
    
    The cube must already have the target shape expected by the network,
    so no padding, cropping, or interpolation is performed.
    
    Parameters
    ----------
    cube : np.ndarray
        Input cube (D, H, W) in original flux units.
        Must match the training target shape.
    model : UNet3DWithPadCrop
        Loaded model. Only ``model.unet`` (the inner UNet3D) is used.
    device : torch.device
    
    Returns
    -------
    np.ndarray
        Denoised cube in original flux units, same shape as input.
    """

    cube_mean, cube_std = cube.mean(), cube.std()
    normalized = (cube - cube_mean) / cube_std # Se normaliza los datos. Esto es necesario debido a que el modelo fue entrenado con datos normalizados.

    tensor_in = torch.tensor(
        normalized[np.newaxis, np.newaxis, ...], dtype=torch.float32
    ).to(device) # Se convierte el cubo en un tensor y se mueve al dispositivo.

    with torch.no_grad():
        out = model.unet(tensor_in)

    return ((out * cube_std) + cube_mean)[0, 0].cpu().numpy() 

# 3. Denoising

## 3.1. Ingreso del Cubo

In [9]:
CUBE_NAME = 'isolated_AC5_N250/isolated_AC5_N250_noisy'


hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
cube = hdu[0].data
cube = np.squeeze(cube)
cube = np.nan_to_num(cube, nan=0.0) # Se reemplazan los NaN por 0
header = hdu[0].header # Header del cubo.


In [4]:
model = load_unet(WEIGHTS_PATH, device) # Cargamos el modelo.

In [11]:
denoised_simple = denoise_simple(cube, model, device) # Denoizamos el cubo.

print(f'[Simple]        shape={denoised_simple.shape}  '
      f'rango=[{denoised_simple.min():.4e}, {denoised_simple.max():.4e}]')

out_path = os.path.join(DATA_DIR, f'{CUBE_NAME}_unet_rawdenoise.fits')
hdu_out = fits.PrimaryHDU(denoised_simple, header=header)
hdu_out.writeto(out_path, overwrite=True, output_verify='fix')
print(f'Guardado: {out_path}')

RuntimeError: Argument #8: Padding size should be less than the corresponding input dimension, but got: padding (1, 1) at dimension 2 of input [1, 128, 1, 12, 12]